In [60]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder.appName("ETL_COVID").getOrCreate()

EXTRACTION


In [4]:
df = spark.read.format('csv').\
        option('inferSchema','true').\
        option('header','true').\
        option('path','COVID_19.csv').\
        load()

In [5]:
df.show(10)

+--------+----+----+----+-----+-------+-----------+------------+--------------------+--------------+----------+---------+------+---------+
|     uid|fips|iso2|iso3|code3| admin2|   latitude|   longitude|      province_state|country_region|      date|confirmed|deaths|recovered|
+--------+----+----+----+-----+-------+-----------+------------+--------------------+--------------+----------+---------+------+---------+
|      16|  60|  AS| ASM|   16|   NULL|    -14.271|    -170.132|      American Samoa|            US|2020-01-22|        0|     0|     NULL|
|     316|  66|  GU| GUM|  316|   NULL|    13.4443|    144.7937|                Guam|            US|2020-01-22|        0|     0|     NULL|
|     580|  69|  MP| MNP|  580|   NULL|    15.0979|    145.6739|Northern Mariana ...|            US|2020-01-22|        0|     0|     NULL|
|     630|  72|  PR| PRI|  630|   NULL|    18.2208|    -66.5901|         Puerto Rico|            US|2020-01-22|        0|     0|     NULL|
|     850|  78|  VI| VIR|  

TRANSFORMATION--

PHASE 1 : DATA PROFILLING

In [7]:
df.count()

450597

In [8]:
df.printSchema()

root
 |-- uid: integer (nullable = true)
 |-- fips: integer (nullable = true)
 |-- iso2: string (nullable = true)
 |-- iso3: string (nullable = true)
 |-- code3: integer (nullable = true)
 |-- admin2: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- province_state: string (nullable = true)
 |-- country_region: string (nullable = true)
 |-- date: date (nullable = true)
 |-- confirmed: integer (nullable = true)
 |-- deaths: integer (nullable = true)
 |-- recovered: integer (nullable = true)



In [9]:
df.select(
    [count(when(col(c).isNull(),c)).alias(c) for c in df.columns]
).show()

+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+
|  uid| fips| iso2| iso3|code3|admin2|latitude|longitude|province_state|country_region|date|confirmed|deaths|recovered|
+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+
|30573|31863|30573|30573|30573| 31476|   13932|    13932|         22575|             0|   0|        0|     0|   420024|
+-----+-----+-----+-----+-----+------+--------+---------+--------------+--------------+----+---------+------+---------+



In [15]:
print("Total records: ",df.count())
print("Duplicate records: ", df.count() - df.distinct().count() )

Total records:  450597
Duplicate records:  0


In [14]:
df.selectExpr(
    "min(date)",
    "max(date)"
).show()

+----------+----------+
| min(date)| max(date)|
+----------+----------+
|2020-01-22|2020-05-29|
+----------+----------+



PHASE 2 : DATA QUALITY CHECK

In [17]:
df.filter((col('confirmed') <0) | (col('deaths') <0 )).count()

0

In [18]:
df.filter(
    col('deaths') > col('confirmed')
).count()

578

In [29]:
silver_layer= df.drop_duplicates()

In [30]:
silver_layer = duplicates_data.withColumn(
    "recovered",
    coalesce(col("recovered"),lit(0))
)

In [31]:
silver_layer.show(5)

+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|   longitude|province_state|country_region|      date|confirmed|deaths|recovered|
+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+
|84001043| 1043|  US| USA|  840|   Cullman|34.13020303|-86.86888037|       Alabama|            US|2020-01-22|        0|     0|        0|
|84001067| 1067|  US| USA|  840|     Henry|31.51148016|-85.24267944|       Alabama|            US|2020-01-22|        0|     0|        0|
|84001123| 1123|  US| USA|  840|Tallapoosa|32.86698258|-85.79833053|       Alabama|            US|2020-01-22|        0|     0|        0|
|84013027|13027|  US| USA|  840|    Brooks|30.83922642|-83.58303423|       Georgia|            US|2020-01-22|        0|     0|        0|
|84021061|21061|  US| USA|  840|  Edmonso

In [32]:
silver_layer = silver_layer.withColumn(
        "active_cases",
        col('confirmed') - col('deaths') - col('recovered')
)

In [39]:
silver_layer.show(5)

+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|   longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|
+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+
|84001043| 1043|  US| USA|  840|   Cullman|34.13020303|-86.86888037|       Alabama|            US|2020-01-22|        0|     0|        0|           0|
|84001067| 1067|  US| USA|  840|     Henry|31.51148016|-85.24267944|       Alabama|            US|2020-01-22|        0|     0|        0|           0|
|84001123| 1123|  US| USA|  840|Tallapoosa|32.86698258|-85.79833053|       Alabama|            US|2020-01-22|        0|     0|        0|           0|
|84013027|13027|  US| USA|  840|    Brooks|30.83922642|-83.58303423|       Georgia|            US|20

In [45]:
silver_layer = silver_layer.withColumn(
    "death_rate",
    when(
        col('confirmed') > 0,
        round(
       (col('deaths') / col('confirmed')) * 100 , 2
    )
    ).otherwise(0)
)

In [46]:
silver_layer.show(5)

+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+----------+
|     uid| fips|iso2|iso3|code3|    admin2|   latitude|   longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|
+--------+-----+----+----+-----+----------+-----------+------------+--------------+--------------+----------+---------+------+---------+------------+----------+
|84001043| 1043|  US| USA|  840|   Cullman|34.13020303|-86.86888037|       Alabama|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84001067| 1067|  US| USA|  840|     Henry|31.51148016|-85.24267944|       Alabama|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84001123| 1123|  US| USA|  840|Tallapoosa|32.86698258|-85.79833053|       Alabama|            US|2020-01-22|        0|     0|        0|           0|       0.0|
|84013027|13027|  US| USA|  840|  

In [47]:
silver_layer = silver_layer.withColumn(
    "recovery_rate",
    when(
        col('confirmed') > 0,
        round(
       (col('recovered') / col('confirmed')) * 100 , 2
    )
    ).otherwise(0)
)

In [51]:
silver_layer.where(col('recovery_rate') > 0).show()

+----+----+----+----+-----+------+--------+---------+--------------------+---------------+----------+---------+------+---------+------------+----------+-------------+
| uid|fips|iso2|iso3|code3|admin2|latitude|longitude|      province_state| country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|
+----+----+----+----+-----+------+--------+---------+--------------------+---------------+----------+---------+------+---------+------------+----------+-------------+
|NULL|NULL|NULL|NULL| NULL|  NULL| 41.1129|  85.2401|            Xinjiang|          China|2020-02-17|       75|     1|       12|          62|      1.33|         16.0|
|NULL|NULL|NULL|NULL| NULL|  NULL| 50.8333|      4.0|                NULL|        Belgium|2020-02-25|        1|     0|        1|           0|       0.0|        100.0|
|NULL|NULL|NULL|NULL| NULL|  NULL| 40.1824| 116.4142|             Beijing|          China|2020-02-02|      191|     1|        9|         181|      0.52|         4.71

In [55]:
# try:
#     silver_layer.write \
#     .mode("overwrite") \
#     .parquet("C:/PySpark_DATA_WRITE/silver/covid")
# except Exception as e:
#     print(e)

PHASE 3 : EXTRACTING KPI's

In [56]:
country_cases = silver_layer.groupBy("country_region").agg(max("confirmed").alias('total_cases')).orderBy(col('total_cases').desc())

In [57]:
country_cases.show(10)

+--------------+-----------+
|country_region|total_cases|
+--------------+-----------+
|            US|    1746019|
|        Brazil|     465166|
|        Russia|     387623|
|United Kingdom|     271222|
|         Spain|     238564|
|         Italy|     232248|
|        France|     183816|
|       Germany|     182922|
|         India|     173491|
|        Turkey|     162120|
+--------------+-----------+
only showing top 10 rows



In [58]:
country_deaths = silver_layer.groupBy("country_region").agg(max('deaths').alias('total_deaths')).orderBy(col('total_deaths').desc())

In [59]:
country_deaths.show(10)

+--------------+------------+
|country_region|total_deaths|
+--------------+------------+
|            US|      102809|
|United Kingdom|       38161|
|         Italy|       33229|
|         Spain|       28752|
|        France|       28663|
|        Brazil|       27878|
|       Belgium|        9430|
|        Mexico|        9415|
|       Germany|        8504|
|          Iran|        7677|
+--------------+------------+
only showing top 10 rows



In [65]:
previous_day_cases = silver_layer.withColumn(
    "previous_day_cases",
    lag('confirmed').over(
        Window.partitionBy(
            'country_region'
        ).orderBy("date")
    )
)

In [66]:
previous_day_cases.show(10)

+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+------------------+
| uid|fips|iso2|iso3|code3|admin2|latitude|longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|previous_day_cases|
+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+------------------+
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-22|        0|     0|        0|           0|       0.0|          0.0|              NULL|
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-23|        0|     0|        0|           0|       0.0|          0.0|                 0|
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-24|        0| 

In [67]:
daily_cases = previous_day_cases.withColumn(
    "daily_new_cases",
    col('confirmed') - col('previous_day_cases')
)

In [81]:
rank_df = daily_cases.withColumn(
    "country_rank",
    dense_rank().over(
        Window.orderBy(col('confirmed').desc())
    )
)

In [82]:
rank_df.show(50)

+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+------------------+---------------+------------+
| uid|fips|iso2|iso3|code3|admin2|latitude|longitude|province_state|country_region|      date|confirmed|deaths|recovered|active_cases|death_rate|recovery_rate|previous_day_cases|daily_new_cases|country_rank|
+----+----+----+----+-----+------+--------+---------+--------------+--------------+----------+---------+------+---------+------------+----------+-------------+------------------+---------------+------------+
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-22|        0|     0|        0|           0|       0.0|          0.0|              NULL|           NULL|        8541|
|NULL|NULL|NULL|NULL| NULL|  NULL|    33.0|     65.0|          NULL|   Afghanistan|2020-01-23|        0|     0|        0|           0|       0.0|          0.0|         